# 03 — Results and robustness charts

**Goal:** compare the trained models and see how their scores drop when the photos are damaged
(blur, JPEG, downscale, noise, screen — see notebook 02).

**This notebook does not train anything.** It reads score files that notebook 05 (Step 9) or Step 1b below
saved, one per experiment:

```
outputs/<experiment>/eval_test.csv      e.g. outputs/03_effnet_finetune/eval_test.csv
```

Each row of that file is one test condition: `clean` (undamaged photos) or one kind of damage at severity 1–5.

**How to use:** this notebook is made for **Kaggle**. Do the settings below once, then run the cells from top to
bottom (Shift+Enter). A GPU is only needed for Step 1b.

### Before you start: Kaggle settings

Open the right-hand panel of the notebook editor (the **>|** icon at the top right if it is hidden):

- **Settings → Internet → On.** Step 1 downloads the project code from GitHub. Kaggle asks you to verify your phone once.
- **Add Input → Datasets →** search **"140k Real and Fake Faces"** (by xhlulu) → **+**. These are the images.
  Kaggle attaches them read-only under `/kaggle/input/`, so nothing is downloaded or unzipped.
- **Add Input → Your Work → Notebooks →** your saved notebook 05 → **+**. This brings in its trained models
  (`outputs/<experiment>/best.pt`) and score files, read-only. Notebook 05 must be saved first with
  **Save Version → Save & Run All (Commit)**.
- Only for Step 1b (scoring models here): **Settings → Accelerator → GPU T4 x2.** Makes the model steps much faster (free, about 30 GPU hours a week).

After you change a setting, Kaggle restarts the session: run the cells again from the top.

## Step 1 — Get the project and find the score files

**What this does, in plain words:**

1. Loads the Python tools we need.
2. Downloads the project code (`src/`, `configs/`) from GitHub into `/kaggle/working/`, the only folder on Kaggle
   where we can write files. If Internet is off, it tells you so in plain words.
3. Finds the dataset you attached with **Add Input** somewhere under `/kaggle/input/`.
4. Makes the list of folders to search: the `outputs/` folder of every notebook you attached with
   **Add Input → Your Work** (for example your saved notebook 05, read-only), and this notebook's own
   `outputs/` folder in `/kaggle/working/`.
5. Finds every `eval_*.csv` file in those folders. If the same file is found twice, the copy in this notebook's
   own `outputs/` wins, because it is the newest.

In [ ]:
# 1. Tools we need
import os
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import yaml
from IPython.display import display

# 2. Get the project code (src/ and configs/) from GitHub.
#    It goes into /kaggle/working: the only folder on Kaggle where we can write files.
REPO_URL = "https://github.com/msondhi/UOT_SCS_3546_041_Deep-Learning.git"
PROJECT_ROOT = Path("/kaggle/working/UOT_SCS_3546_041_Deep-Learning")

if not PROJECT_ROOT.exists():
    result = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT_ROOT)])
    if result.returncode != 0:
        raise RuntimeError("Could not download the project from GitHub. "
                           "Turn on Settings -> Internet in the right-hand panel, then run this cell again.")
sys.path.insert(0, str(PROJECT_ROOT / "src"))   # lets Python import the project's own code
print("Project folder:", PROJECT_ROOT)

# 3. Find the dataset you attached with Add Input. Kaggle puts it somewhere under /kaggle/input.
def find_dataset():
    """Look inside /kaggle/input for the folder called real-vs-fake that has train/ valid/ test/."""
    for depth in range(1, 5):                                  # look 1, 2, 3, then 4 folders deep
        pattern = "/".join(["*"] * depth) + "/real-vs-fake"    # e.g. "*/*/real-vs-fake"
        for folder in Path("/kaggle/input").glob(pattern):
            if (folder / "train").is_dir():
                return folder
    return None

DATA_ROOT = find_dataset()       # folder with train/ valid/ test/, or None if not attached
if DATA_ROOT is None:
    print('Dataset not found: Add Input -> Datasets -> "140k Real and Fake Faces", then run this cell again.')
else:
    print("Images folder:", DATA_ROOT)

# 4. Folders to search for trained models and scores:
#  - outputs/ of other notebooks you attached with Add Input -> Your Work (read-only),
#    e.g. your saved notebook 05: /kaggle/input/<notebook>/.../outputs/
#  - this notebook's own outputs/ in /kaggle/working (listed last, so a new file made here wins)
RESULTS_FOLDERS = []
for depth in range(1, 5):
    RESULTS_FOLDERS += sorted(Path("/kaggle/input").glob("/".join(["*"] * depth) + "/outputs"))
RESULTS_FOLDERS.append(PROJECT_ROOT / "outputs")

# 5. Find every eval_*.csv file: <results folder>/<experiment>/eval_*.csv
def find_eval_files():
    found = {}
    for results_folder in RESULTS_FOLDERS:
        for file in sorted(results_folder.glob("*/eval_*.csv")):
            found[(file.parent.name, file.name)] = file   # a later folder replaces an older copy
    return sorted(found.values())

eval_files = find_eval_files()
print(f"Found {len(eval_files)} score file(s):")
for file in eval_files:
    print("  ", file)

## Step 1b (optional) — Score models with `evaluate.py` here

Skip this if Step 1 already found a score file for every model (notebook 05, Step 9 makes them too).

**What `evaluate.py` does, in plain words:**

1. Opens a trained model (`best.pt`) and the settings saved inside it.
2. Makes a list of test conditions:
   - always `clean`: the normal test photos, undamaged (saved as severity 0);
   - only if you ask for damage: each of the 5 kinds of damage at severities 1–5, which adds 25 more conditions.
3. For each condition, it runs the model on the test photos (damaged on the fly, the files are not changed)
   and calculates accuracy, ROC AUC, FAR, FRR, …
4. Saves one row per condition into `eval_test.csv`.

So a file with **1 row** was tested on clean photos only. A file with **26 rows** was tested on clean + all damage.

**How this cell decides what to test (`INCLUDE_DAMAGE = "auto"`):** the robustness comparison in the project plan
is **E3 (standard training) vs. E6 (robust training)**, so those two are tested on clean + damaged photos and the
other models on clean photos only:

| Model | How it was trained (`augment.degradation`) | What this cell tests |
|---|---|---|
| `03_effnet_finetune` (E3, standard training) | clean photos only (`false`) | clean + all damage (26 tests) |
| `06_effnet_robust` (E6, robust training) | clean + randomly damaged photos (`true`) | clean + all damage (26 tests) |
| `01`, `02`, `04`, `05` | clean photos only (`false`) | clean test photos only (1 test) |

You can override it with `INCLUDE_DAMAGE = True` (every model gets all 26 tests) or `False` (clean only).

**Other controls:** which models (`CHECKPOINTS`) and how many test photos (`TEST_PHOTOS`, fewer = faster).
It needs the dataset attached and is much faster with a GPU (Settings → Accelerator).
Models from an attached notebook are read-only, so their scores are saved in **this** notebook's
`outputs/<experiment>/`. Change `RUN_EVALUATION` to `True` to run it.

In [ ]:
RUN_EVALUATION = False      # change to True to run evaluate.py
INCLUDE_DAMAGE = "auto"     # "auto" = damage tests for E3 and E6 only; True = always add damage; False = clean only
ROBUSTNESS_MODELS = ["03_effnet_finetune", "06_effnet_robust"]   # the E3 vs. E6 robustness comparison
TEST_PHOTOS = 2000          # use this many test photos to save time; None = all 20,000
CHECKPOINTS = None          # None = every best.pt found; or a list of paths to best.pt files

if RUN_EVALUATION and DATA_ROOT is None:
    print("Cannot score: attach the dataset first (see Step 1).")
elif RUN_EVALUATION:
    from deepfake_kyc.evaluate import main as run_evaluate

    if CHECKPOINTS is None:
        CHECKPOINTS = []
        for results_folder in RESULTS_FOLDERS:
            CHECKPOINTS += sorted(results_folder.glob("*/best.pt"))
    if not CHECKPOINTS:
        print("No best.pt found: train a model in notebook 05 and attach it (Add Input -> Your Work).")

    for checkpoint in map(Path, CHECKPOINTS):      # accepts paths or plain text
        # How was this model trained? The training settings are saved next to best.pt.
        settings = yaml.safe_load((checkpoint.parent / "config.yaml").read_text())
        trained_with_damage = settings.get("augment", {}).get("degradation", False)
        experiment = checkpoint.parent.name.removesuffix("_notebook_demo")   # quick runs count too
        test_damage = (experiment in ROBUSTNESS_MODELS) if INCLUDE_DAMAGE == "auto" else INCLUDE_DAMAGE

        print("=" * 70)
        print("Model:     ", checkpoint.parent.name)
        print("Trained on:", "clean + randomly damaged photos" if trained_with_damage else "clean photos only")
        print("Testing on:", "clean + damaged photos (26 tests)" if test_damage else "clean photos only (1 test)")

        # The same words you would type after "python -m deepfake_kyc.evaluate" in a terminal.
        # data.root tells evaluate.py where the test images are (the attached dataset).
        arguments = ["--checkpoint", str(checkpoint), "--set", f"data.num_workers={os.cpu_count()}", f"data.root={DATA_ROOT.as_posix()}"]
        if TEST_PHOTOS is not None:
            arguments.append(f"data.test_subset={TEST_PHOTOS}")
        if test_damage:
            arguments += ["--degradation", "all"]

        # Models from another notebook are read-only, so save the score in this notebook's outputs/.
        score_folder = PROJECT_ROOT / "outputs" / checkpoint.parent.name
        score_folder.mkdir(parents=True, exist_ok=True)
        if not (score_folder / "config.yaml").exists():
            (score_folder / "config.yaml").write_text((checkpoint.parent / "config.yaml").read_text())
        arguments += ["--output", str(score_folder / "eval_test.csv")]

        run_evaluate(arguments)

    eval_files = find_eval_files()   # look again so the new score files are included
    print(f"Now {len(eval_files)} score file(s) found.")
else:
    print("Skipped. Change RUN_EVALUATION to True to score models from this notebook.")

## Step 2 — Put all score files into one table

**What this does:** reads every file and stacks them into one table called `results`.
It adds a `run` column so we know which experiment each row came from, e.g. `03_effnet_finetune (test)`.
The folder name is the experiment name; the part after `eval_` in the file name is the test set (`test` or e.g. `ood`).

In [ ]:
tables = []
for file in eval_files:
    table = pd.read_csv(file)
    experiment = file.parent.name                    # e.g. 03_effnet_finetune
    test_set = file.stem.replace("eval_", "")        # e.g. test
    table["run"] = f"{experiment} ({test_set})"

    # How was this model trained? Read the settings saved next to the score file.
    settings_file = file.parent / "config.yaml"
    settings = yaml.safe_load(settings_file.read_text()) if settings_file.exists() else {}
    trained_with_damage = settings.get("augment", {}).get("degradation", False)
    table["trained_on"] = "clean + damaged photos" if trained_with_damage else "clean photos only"
    tables.append(table)

if tables:
    results = pd.concat(tables, ignore_index=True)
    print(f"{len(results)} rows from {len(tables)} file(s). First rows:")
    display(results.head())
else:
    results = pd.DataFrame()
    print("No score files yet: attach your saved notebook 05, or run Step 1b.")

## Step 2b — What was tested in each file?

**What this does:** for each run, says how the model was **trained** (clean only, or clean + damaged photos)
and what it was **tested** on (clean only, or clean + which kinds of damage and severities).
Use this to understand why a run does or does not appear in the charts of Steps 4–5.

In [ ]:
if not results.empty:
    for run in results["run"].unique():
        rows = results[results["run"] == run]
        damaged = rows[rows["condition"] != "clean"]
        print(f"{run}: {len(rows)} test condition(s)")
        print(f"   trained on:     {rows['trained_on'].iloc[0]}")
        if (rows["condition"] == "clean").any():
            print("   clean photos:   yes")
        if damaged.empty:
            print("   damaged photos: no  -> clean only, so this run is skipped in Steps 4-5")
        else:
            for damage in sorted(damaged["condition"].unique()):
                severities = sorted(damaged.loc[damaged["condition"] == damage, "severity"])
                print(f"   damaged photos: {damage:<9} severities {severities}")
        print()
else:
    print("Skipped: no results loaded.")

## Step 3 — Clean test results: which model is best?

**What this does:** shows one row per run, using only the `clean` (undamaged) photos, best first.

**What the columns mean** (fake = the thing we want to catch):

| Column | Plain meaning | Better when |
|---|---|---|
| `accuracy` | share of all photos labelled correctly | higher |
| `precision` | of the photos flagged as fake, how many really were fake | higher |
| `recall` | of all the fakes, how many were caught | higher |
| `f1` | one number that balances precision and recall | higher |
| `roc_auc` | how well the model ranks fakes above reals, at any threshold (1.0 = perfect, 0.5 = guessing) | higher |
| `far` | **False Acceptance Rate**: share of fakes that got through as real (the fraud risk) | lower |
| `frr` | **False Rejection Rate**: share of real customers wrongly flagged as fake (the annoyed-customer risk) | lower |

In [ ]:
if not results.empty:
    clean = results[results["condition"] == "clean"]
    clean = clean.sort_values("roc_auc", ascending=False)
    columns = ["run", "accuracy", "precision", "recall", "f1", "roc_auc", "far", "frr"]
    display(clean[columns].round(3))
else:
    print("Skipped: no results loaded.")

## Step 4 — Robustness charts: how much do scores drop on damaged photos?

**What this does:** draws one small chart per kind of damage. The x-axis is the severity
(0 = clean photos, 1 = a little damage … 5 = a lot). Each line is one run.

**How to read it:** a line that stays flat means the model copes well with that damage.
A line that falls steeply means the model breaks on poor-quality uploads.

**Try it:** change `metric` to `"accuracy"`, `"far"` or `"frr"` and run the cell again.
For `far` and `frr`, lower is better, so a rising line is the bad sign.

In [ ]:
metric = "roc_auc"    # try "accuracy", "far" or "frr"

if not results.empty and (results["condition"] != "clean").any():
    damage_names = sorted(results.loc[results["condition"] != "clean", "condition"].unique())

    fig, axes = plt.subplots(1, len(damage_names), figsize=(4 * len(damage_names), 3.5),
                             sharey=True, squeeze=False)
    for ax, damage in zip(axes[0], damage_names):
        for run in results["run"].unique():
            # this run's rows for this damage, plus its clean row as severity 0
            rows = results[(results["run"] == run) & results["condition"].isin(["clean", damage])]
            rows = rows.sort_values("severity")
            if len(rows) > 1:
                ax.plot(rows["severity"], rows[metric], marker="o", label=run)
        ax.set_title(damage)
        ax.set_xlabel("severity (0 = clean)")
        ax.set_xticks(range(6))
    axes[0, 0].set_ylabel(metric)
    axes[0, -1].legend(fontsize=8)
    fig.suptitle(f"{metric} as the damage gets worse")
    plt.tight_layout()
    plt.show()
else:
    print("Skipped: no damaged-photo results yet.")

## Step 5 — Summary: clean score vs. worst-case score

**What this does:** for each run and each kind of damage, compares the clean score with the score at severity 5,
and shows how much it dropped. This gives numbers you can quote in the report,
e.g. *"blur at severity 5 lowers ROC AUC by 0.12"*.

In [ ]:
if not results.empty and (results["condition"] != "clean").any():
    summary_rows = []
    for run in results["run"].unique():
        run_rows = results[results["run"] == run]
        clean_rows = run_rows[run_rows["condition"] == "clean"]
        if clean_rows.empty:
            continue
        clean_score = clean_rows[metric].iloc[0]
        for damage in sorted(run_rows.loc[run_rows["condition"] != "clean", "condition"].unique()):
            damage_rows = run_rows[run_rows["condition"] == damage]
            worst = damage_rows.loc[damage_rows["severity"].idxmax()]   # highest severity tested
            summary_rows.append({
                "run": run,
                "damage": damage,
                f"clean {metric}": clean_score,
                f"{metric} at severity {int(worst['severity'])}": worst[metric],
                "change": worst[metric] - clean_score,
            })
    display(pd.DataFrame(summary_rows).round(3))
else:
    print("Skipped: no damaged-photo results yet.")

## Step 6 — One overall score per model (clean + damaged mix)

**What this does:** gives each model one score for each column, matching how it was tested:

- Model tested on **clean + damaged** photos (E3 and E6 by default): the **average** over all its
  26 tests (1 clean + 5 kinds of damage × 5 severities). This is like scoring it on a test set where
  photos come in every quality, from perfect to badly damaged.
- Model tested on **clean photos only**: its clean score.

The `tested on` column says which one you are looking at. Only compare models that have the same `tested on` value:
an average that includes damaged photos is naturally lower than a clean-only score.

In [ ]:
if not results.empty:
    overall_rows = []
    for run in results["run"].unique():
        run_rows = results[results["run"] == run]
        has_damage = (run_rows["condition"] != "clean").any()
        overall_rows.append({
            "run": run,
            "trained on": run_rows["trained_on"].iloc[0],
            "tested on": f"clean + damaged ({len(run_rows)} tests)" if has_damage else "clean only (1 test)",
            "accuracy": run_rows["accuracy"].mean(),   # mean = average over all tests of this run
            "roc_auc": run_rows["roc_auc"].mean(),
            "far": run_rows["far"].mean(),
            "frr": run_rows["frr"].mean(),
        })
    overall = pd.DataFrame(overall_rows).sort_values("roc_auc", ascending=False)
    display(overall.round(3))
else:
    print("Skipped: no results loaded.")

## Next

Notebook **04_error_analysis** opens one trained model, finds the photos it got wrong,
and shows heatmaps (Grad-CAM) of where the model was looking.

## Quick notes to remember

- These numbers come from the **test** set. Use them only for the final comparison, not to tune settings.
- `far` matters most for fraud (fakes getting through); `frr` matters most for customer experience.
- A run tested on clean photos only appears in Steps 3 and 6 but not in Steps 4–5.
- Which models train on damaged photos is set in the config: `augment.degradation: false` in `configs/base.yaml`,
  so `01`–`05` train on clean photos only. `06_effnet_robust` (E6) turns it on: robust training.
- The key robustness figure is the Step 4 chart of **E3 (standard) vs. E6 (robust training)**; both are
  tested on damaged photos by default (Step 1b).
- Files made here (new score files) are kept only when you click **Save Version**.